# Exploration complète de `FrequencyConverter`

Ce notebook illustre le comportement des cinq méthodes publiques du `FrequencyConverter` :

1. **`get_conversion_factor`** — facteur de conversion approximatif entre deux fréquences
2. **`convert`** — implémentation de `TemporalConverter.convert()`, point d'entrée générique
3. **`convert_frequency`** — méthode principale de conversion avec gestion automatique up/downsampling
4. **`aggregate_to_lower_frequency`** — agrégation (downsampling) avec contrôle fin des périodes incomplètes
5. **`interpolate_to_higher_frequency`** — interpolation (upsampling) avec contrôle de la limite et de la direction

Chaque méthode est testée sur des **séries temporelles** et des **données de panel**.

> **Contrat d'index** : l'index de sortie du `FrequencyConverter` porte toujours la fréquence cible
> (ou l'union des index cibles pour un `target_freq` dict). Pour construire des jeux de données
> d'entraînement dont l'index d'origine est préservé ou densifié, voir le `FrequencyAligner`
> (`tsforecast.frequency.frequency_aligner`).


## 0. Imports & utilitaires

In [ ]:
# Importation des modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import sys

# Ajout du chemin
sys.path.append('..')

# Importation du FrequencyConverter
from tsforecast.utils.frequency.converter import FrequencyConverter

converter = FrequencyConverter()

# Palette de couleurs
COLORS = {
    'original': '#4e79a7',
    'aggregated': '#e15759',
    'interpolated': '#59a14f',
    'nan': '#bab0ac',
    'ffill': '#f28e2b',
    'bfill': '#b07aa1',
    'converted': '#76b7b2',
    'aligned': '#edc948',
}


def print_section(title: str) -> None:
    """Print a formatted section header."""
    print(f"\n{'─' * 60}")
    print(f"  {title}")
    print(f"{'─' * 60}")


def show_series(s: pd.Series, name: str = "") -> None:
    """Print a series with its metadata."""
    label = f" ({name})" if name else ""
    freq = getattr(s.index, 'inferred_freq', None) or pd.infer_freq(s.index)
    print(f"  Fréquence détectée : {freq} | {len(s)} points | {s.isna().sum()} NaN{label}")
    print(s.to_string(float_format='{:.2f}'.format))

---
## 1. `get_conversion_factor`

Retourne un facteur de conversion **approximatif** entre deux fréquences. Ce facteur est utilisé en interne pour déterminer le nombre de sous-périodes attendues (par exemple 3 mois par trimestre).

### 1.1 Séries temporelles — Facteurs classiques

In [ ]:
print_section("Facteurs de conversion entre fréquences courantes")

pairs = [
    ('daily', 'monthly'),
    ('monthly', 'quarterly'),
    ('quarterly', 'annual'),
    ('daily', 'weekly'),
    ('monthly', 'annual'),
    ('weekly', 'monthly'),
]

for from_freq, to_freq in pairs:
    factor = converter.get_conversion_factor(from_freq, to_freq)
    print(f"  {from_freq:>12s} → {to_freq:<12s} : {factor:.1f}")

### 1.2 Interprétation

Le facteur indique combien de périodes source composent une période cible :
- `monthly → quarterly = 3.0` signifie qu'il faut 3 mois pour former 1 trimestre
- `daily → monthly ≈ 30.0` est une approximation (les mois ont entre 28 et 31 jours)

---
## 2. `convert` — Point d'entrée générique

`convert(value, from_unit, to_unit, **kwargs)` est l'implémentation de la méthode abstraite `TemporalConverter.convert()`. Elle redirige vers `convert_frequency` en ignorant `from_unit` (la fréquence source est auto-détectée).

### 2.1 Séries temporelles

In [ ]:
# Création d'une série journalière
dates_d = pd.date_range('2024-01-01', periods=90, freq='D')
series_d = pd.Series(np.sin(np.linspace(0, 2 * np.pi, 90)) * 50 + 100, index=dates_d, name='signal')

# Affichage
print("Série journalière originale :")
print(f"  {len(series_d)} points | {series_d.index[0].date()} → {series_d.index[-1].date()}")

# Conversion vers mensuel via convert
result_m = converter.convert(series_d, from_unit='daily', to_unit='monthly', method='mean')
print("\nRésultat mensuel (convert, method='mean') :")
show_series(result_m, "mensuel")

### 2.2 Données de panel

In [ ]:
# Création d'un panel journalier avec 2 entités
rng = np.random.default_rng(seed=42)
dates_panel_d = pd.date_range('2024-01-01', periods=60, freq='D')

frames = []
for entity in ['France', 'Allemagne']:
    df_ent = pd.DataFrame({
        'exports': rng.normal(100, 10, 60).cumsum() / 60 + 100,
        'imports': rng.normal(80, 8, 60).cumsum() / 60 + 80,
    }, index=dates_panel_d)
    df_ent['pays'] = entity
    frames.append(df_ent)

df_panel_d = pd.concat(frames).set_index('pays', append=True).swaplevel()
df_panel_d.index.names = ['pays', 'date']

print("Panel journalier :")
print(df_panel_d.head(5))
print(f"  ...\n  {df_panel_d.shape[0]} lignes | entités : {df_panel_d.index.get_level_values('pays').unique().tolist()}")

# Conversion vers mensuel via convert
result_panel_m = converter.convert(df_panel_d, from_unit='daily', to_unit='monthly', method='mean')
print("\nRésultat mensuel (panel) :")
print(result_panel_m)

---
## 3. `convert_frequency` — Méthode principale

`convert_frequency` est la méthode la plus complète. Elle détecte automatiquement la fréquence source, détermine si la conversion est un upsampling ou un downsampling, et gère les positions `S`/`E`.

### 3.1 Séries temporelles — Downsampling et upsampling

In [ ]:
# Série mensuelle sur 2 ans
dates_m = pd.date_range('2024-01-01', periods=24, freq='MS')
series_m = pd.Series(
    np.linspace(100, 200, 24) + rng.normal(0, 5, 24),
    index=dates_m, name='indicateur'
)

# Downsampling : mensuel → trimestriel
quarterly = converter.convert_frequency(series_m, target_freq='quarterly', method='mean')
print("Downsampling mensuel → trimestriel (mean) :")
show_series(quarterly)

# Upsampling : trimestriel → mensuel
back_to_monthly = converter.convert_frequency(quarterly, target_freq='monthly', method='linear')
print("\nUpsampling trimestriel → mensuel (linear) :")
show_series(back_to_monthly)

In [ ]:
# Visualisation aller-retour
fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(series_m.index, series_m.values, 'o-', color=COLORS['original'],
        markersize=4, alpha=0.6, label='Mensuel original')
ax.plot(quarterly.index, quarterly.values, 's-', color=COLORS['aggregated'],
        markersize=8, label='Trimestriel (agrégé)')
ax.plot(back_to_monthly.index, back_to_monthly.values, 'x--', color=COLORS['interpolated'],
        markersize=6, alpha=0.8, label='Mensuel (ré-interpolé)')

ax.set_title('Aller-retour : mensuel → trimestriel → mensuel')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 3.2 Séries temporelles — `target_freq` avec position explicite

In [ ]:
# Comparaison QS vs QE
print_section("convert_frequency avec position explicite")

for target in ['QS', 'QE']:
    result = converter.convert_frequency(series_m, target_freq=target, method='mean')
    print(f"\n  target_freq='{target}' :")
    for date, val in result.items():
        print(f"    {date.strftime('%Y-%m-%d')} → {val:.2f}")

### 3.3 DataFrame — `target_freq` sous forme de dictionnaire

Avec un `target_freq` dict, l'index de sortie est **l'union des index cibles** des colonnes
converties : l'index source disparaît dès que toutes les colonnes sont converties. Seules les
colonnes absentes du dictionnaire (ou déjà à leur fréquence cible) conservent leurs dates
d'origine dans l'union. Les NaN créés par l'union de fréquences mixtes sont comblés selon
`alignment_method`.


In [ ]:
# DataFrame avec colonnes à fréquences cibles différentes
dates_d2 = pd.date_range('2024-01-01', periods=180, freq='D')
df_multi = pd.DataFrame({
    'ventes_jour': rng.normal(100, 10, 180),
    'temperature': np.sin(np.linspace(0, 2 * np.pi, 180)) * 15 + 20,
}, index=dates_d2)

# Affichage
print("DataFrame journalier :")
print(df_multi.head())

# Conversion : ventes_jour → mensuel, temperature → hebdomadaire
result_dict = converter.convert_frequency(
    df_multi,
    target_freq={'ventes_jour': 'monthly', 'temperature': 'weekly'},
    method='mean',
    alignment_method='ffill',
)

# L'index de sortie est l'union des fins de mois et des fins de semaine :
# l'index journalier d'origine a disparu
print("\nRésultat avec target_freq dict (union mensuel ∪ hebdo, aligné ffill) :")
print(result_dict.head(15))
print(f"\n{len(df_multi)} lignes journalières → {len(result_dict)} lignes (union des index cibles)")


### 3.4 Données de panel

In [ ]:
# Panel mensuel avec 2 entités
dates_pm = pd.date_range('2024-01-01', periods=12, freq='MS')

frames_pm = []
for entity in ['secteur_A', 'secteur_B']:
    offset = 0 if entity == 'secteur_A' else 50
    df_e = pd.DataFrame({
        'ca': np.linspace(100, 150, 12) + offset + rng.normal(0, 3, 12),
    }, index=dates_pm)
    df_e['secteur'] = entity
    frames_pm.append(df_e)

df_panel_m = pd.concat(frames_pm).set_index('secteur', append=True).swaplevel()
df_panel_m.index.names = ['secteur', 'date']

print("Panel mensuel :")
print(df_panel_m)

# Conversion vers trimestriel
result_panel_q = converter.convert_frequency(df_panel_m, target_freq='quarterly', method='sum')
print("\nPanel trimestriel (sum) :")
print(result_panel_q)

---
## 4. `aggregate_to_lower_frequency` — Agrégation détaillée

Cette section explore en détail les paramètres de l'agrégation :
- **`method`** : `mean`, `sum`, `first`, `last`, `min`, `max`, `median`, `std`, `count`
- **`full_periods_only`** : masquage des périodes incomplètes

### 4.1 Cas nominal : agrégation mensuelle → trimestrielle complète

In [ ]:
# Série mensuelle sur 12 mois (4 trimestres complets)
dates_m12 = pd.date_range('2024-01-01', periods=12, freq='MS')
values_12 = [100, 110, 105, 120, 115, 125, 130, 140, 135, 150, 145, 155]
series_m12 = pd.Series(values_12, index=dates_m12, name='indicateur')

# Affichage
print("Série mensuelle originale :")
print(series_m12.to_frame().T.to_string())

print_section("Agrégation mensuelle → trimestrielle (différentes méthodes)")

for method in ['mean', 'sum', 'last', 'first', 'min', 'max']:
    result = converter.aggregate_to_lower_frequency(series_m12, 'QS', method=method)
    vals = '  |  '.join(f"{d.strftime('%Y-%m-%d')}: {v:.1f}" for d, v in result.items())
    print(f"  {method:>6s} : {vals}")

In [ ]:
# Visualisation comparative des méthodes
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=False)

for ax, method in zip(axes, ['mean', 'sum', 'last']):
    result = converter.aggregate_to_lower_frequency(series_m12, 'QS', method=method)

    ax.bar(series_m12.index, series_m12.values, width=20, alpha=0.4,
           color=COLORS['original'], label='Mensuel (original)')
    ax.bar(result.index, result.values, width=60, alpha=0.6,
           color=COLORS['aggregated'], label=f'Trimestriel ({method})')

    ax.set_title(f"method='{method}'")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('Agrégation mensuelle → trimestrielle (4 trimestres complets)', fontsize=12)
plt.tight_layout()
plt.show()

### 4.2 Cas critique : trimestre incomplet (2 mois sur 3) et `full_periods_only`

On crée une série mensuelle de **14 mois** (janvier 2024 → février 2025) : le Q1 2025 ne contient que **2 mois** (janvier et février) sur les 3 attendus.

- Avec `full_periods_only=False` (défaut) : le trimestre incomplet est agrégé sur les 2 mois disponibles
- Avec `full_periods_only=True` : le trimestre incomplet produit `NaN`

In [ ]:
# Série mensuelle avec trimestre incomplet en fin de période
dates_14 = pd.date_range('2024-01-01', periods=14, freq='MS')  # Jan 2024 → Fév 2025
values_14 = list(range(100, 114))
series_14 = pd.Series(values_14, index=dates_14, name='indicateur')

print("Série mensuelle (14 mois, dernier trimestre incomplet) :")
print(series_14)

print_section("Comparaison full_periods_only=False vs True")

for fpo in [False, True]:
    print(f"\n  full_periods_only={fpo} :")
    for method in ['mean', 'sum', 'count']:
        result = converter.aggregate_to_lower_frequency(
            series_14, 'QS', method=method, full_periods_only=fpo
        )
        vals = '  |  '.join(
            f"{d.strftime('%Y-%m-%d')}: {v:.1f}" if not np.isnan(v) else f"{d.strftime('%Y-%m-%d')}: NaN"
            for d, v in result.items()
        )
        print(f"    {method:>6s} : {vals}")

In [ ]:
# Visualisation : trimestre complet vs incomplet
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, fpo in zip(axes, [False, True]):
    result_mean = converter.aggregate_to_lower_frequency(series_14, 'QS', method='mean', full_periods_only=fpo)
    result_sum = converter.aggregate_to_lower_frequency(series_14, 'QS', method='sum', full_periods_only=fpo)

    # Barres mensuelles (couleur différente pour les mois du trimestre incomplet)
    colors_monthly = [COLORS['original']] * 12 + [COLORS['ffill']] * 2
    ax.bar(series_14.index, series_14.values, width=20, alpha=0.4, color=colors_monthly)

    # Barres trimestrielles
    valid_mean = result_mean.dropna()
    nan_mean = result_mean[result_mean.isna()]
    ax.bar(valid_mean.index, valid_mean.values, width=60, alpha=0.5,
           color=COLORS['aggregated'], label='mean (trimestre)')
    if len(nan_mean) > 0:
        ax.bar(nan_mean.index, [0] * len(nan_mean), width=60, alpha=0.3,
               color=COLORS['nan'], label='NaN (incomplet)')

    ax.set_title(f"full_periods_only={fpo}")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('Impact de full_periods_only sur le trimestre incomplet (2/3 mois)', fontsize=12)
plt.tight_layout()
plt.show()

### 4.3 Analyse détaillée du trimestre incomplet

| Paramètre | `full_periods_only=False` | `full_periods_only=True` |
|---|---|---|
| `mean` Q1 2025 | Moyenne sur 2 mois (correct mais partiel) | `NaN` |
| `sum` Q1 2025 | Somme de 2 mois (**sous-estimée**) | `NaN` |
| `count` Q1 2025 | 2 (permet de détecter l'incomplétude) | `NaN` |

**Recommandation** : pour les agrégations en `sum`, utiliser `full_periods_only=True` afin d'éviter les sous-estimations silencieuses.

### 4.4 NaN en début de période et délais de publication en fin de période

Scénario réaliste :
- Les **2 premiers mois** ont des NaN (données pas encore disponibles historiquement)
- Les **2 derniers mois** ont des NaN (délais de publication)

In [ ]:
# Série mensuelle avec NaN en début et fin
dates_m_nan = pd.date_range('2024-01-01', periods=18, freq='MS')  # Jan 2024 → Jun 2025
values_nan = (
    [np.nan, np.nan]                        # NaN en début (jan-fév 2024)
    + list(range(100, 114))                 # Données valides (mar 2024 → avr 2025)
    + [np.nan, np.nan]                      # NaN en fin = délais de publication (mai-jun 2025)
)
series_m_nan = pd.Series(values_nan, index=dates_m_nan, name='indicateur')

print("Série mensuelle avec NaN (début + fin) :")
print(series_m_nan)

print_section("Agrégation avec NaN : full_periods_only=False vs True")

for fpo in [False, True]:
    print(f"\n  full_periods_only={fpo} :")
    for method in ['mean', 'sum', 'count']:
        result = converter.aggregate_to_lower_frequency(
            series_m_nan, 'QS', method=method, full_periods_only=fpo
        )
        vals = []
        for d, v in result.items():
            if np.isnan(v):
                vals.append(f"{d.strftime('%Y-%m-%d')}: NaN")
            else:
                vals.append(f"{d.strftime('%Y-%m-%d')}: {v:.1f}")
        print(f"    {method:>6s} : {'  |  '.join(vals)}")

In [ ]:
# Visualisation : impact des NaN sur l'agrégation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, fpo in zip(axes, [False, True]):
    result = converter.aggregate_to_lower_frequency(
        series_m_nan, 'QS', method='mean', full_periods_only=fpo
    )
    count = converter.aggregate_to_lower_frequency(
        series_m_nan, 'QS', method='count', full_periods_only=fpo
    )

    # Barres mensuelles
    colors_m = []
    for v in series_m_nan.values:
        colors_m.append(COLORS['nan'] if np.isnan(v) else COLORS['original'])
    ax.bar(series_m_nan.index, series_m_nan.fillna(0).values, width=20, alpha=0.4, color=colors_m)

    # Barres trimestrielles
    for d, v in result.items():
        color = COLORS['aggregated'] if not np.isnan(v) else COLORS['nan']
        bar_val = v if not np.isnan(v) else 0
        alpha = 0.6 if not np.isnan(v) else 0.2
        ax.bar(d, bar_val, width=60, alpha=alpha, color=color)

    ax.set_title(f"full_periods_only={fpo}")
    ax.grid(True, alpha=0.3)

    # Annotation du count
    for d, c in count.items():
        if not np.isnan(c):
            ax.annotate(f"n={int(c)}", xy=(d, 5), fontsize=7, ha='center', color='gray')

# Légende commune
patches = [
    mpatches.Patch(color=COLORS['original'], alpha=0.4, label='Mensuel (valide)'),
    mpatches.Patch(color=COLORS['nan'], alpha=0.4, label='Mensuel (NaN)'),
    mpatches.Patch(color=COLORS['aggregated'], alpha=0.6, label='Trimestriel (agrégé)'),
]
fig.legend(handles=patches, loc='lower center', ncol=3, fontsize=9)
fig.suptitle("Agrégation avec NaN en début de période et délais de publication", fontsize=12)
plt.tight_layout(rect=[0, 0.08, 1, 0.95])
plt.show()

### 4.5 Données de panel

In [ ]:
# Panel mensuel avec 2 entités, trimestres incomplets asymétriques
dates_pm2 = pd.date_range('2024-01-01', periods=14, freq='MS')

# Entité A : 14 mois (trimestre incomplet)
df_ea = pd.DataFrame({
    'ventes': rng.normal(100, 10, 14).cumsum() / 14 + 100,
    'cout': rng.normal(50, 5, 14).cumsum() / 14 + 50,
}, index=dates_pm2)
df_ea['entity'] = 'entité_A'

# Entité B : 14 mois avec NaN en début
vals_b_v = [np.nan, np.nan] + list(rng.normal(200, 15, 12).cumsum() / 12 + 200)
vals_b_c = [np.nan, np.nan] + list(rng.normal(90, 7, 12).cumsum() / 12 + 90)
df_eb = pd.DataFrame({
    'ventes': vals_b_v,
    'cout': vals_b_c,
}, index=dates_pm2)
df_eb['entity'] = 'entité_B'

# Assemblage du panel
df_panel_agg = pd.concat([df_ea, df_eb]).set_index('entity', append=True).swaplevel()
df_panel_agg.index.names = ['entity', 'date']

print("Panel mensuel (14 mois, entité_B avec NaN en début) :")
print(df_panel_agg.head(5))
print("  ...")
print(df_panel_agg.tail(5))

# Agrégation par entité
print_section("Agrégation du panel mensuel → trimestriel")

for entity in ['entité_A', 'entité_B']:
    sub = df_panel_agg.xs(entity, level='entity')
    for method, fpo in [('mean', False), ('sum', True)]:
        result = converter.aggregate_to_lower_frequency(sub, 'QS', method=method, full_periods_only=fpo)
        print(f"\n  {entity} — {method} (full_periods_only={fpo}) :")
        for date, row in result.iterrows():
            v_str = f"{row['ventes']:.2f}" if not np.isnan(row['ventes']) else "NaN"
            c_str = f"{row['cout']:.2f}" if not np.isnan(row['cout']) else "NaN"
            print(f"    {date.strftime('%Y-%m-%d')} → ventes={v_str}, cout={c_str}")

---
## 5. `interpolate_to_higher_frequency` — Interpolation détaillée

Cette section explore les paramètres de l'interpolation :
- **`method`** : `linear`, `nearest`, `cubic`, `quadratic`, `zero`, `slinear`, etc.
- **`limit`** : nombre max de NaN consécutifs à remplir (`'default'` = facteur de conversion, `None` = illimité)
- **`limit_direction`** : `'forward'`, `'backward'`, `'both'`
- **`limit_area`** : `'inside'` (entre valeurs connues), `'outside'` (aux extrémités)
- **`target_position`** : position de la fréquence cible (`'S'` ou `'E'`)

### 5.1 Cas nominal : interpolation trimestrielle → mensuelle

In [ ]:
# Série trimestrielle sur 2 ans
dates_q = pd.date_range('2024-01-01', periods=8, freq='QS')
values_q = [100, 120, 115, 130, 125, 140, 135, 150]
series_q = pd.Series(values_q, index=dates_q, name='pib')

print("Série trimestrielle originale :")
print(series_q)

print_section("Interpolation trimestrielle → mensuelle (différentes méthodes)")

for method in ['linear', 'nearest', 'cubic', 'zero']:
    result = converter.interpolate_to_higher_frequency(series_q, 'MS', method=method)
    nan_count = result.isna().sum()
    print(f"  {method:>10s} : {len(result)} points | {nan_count} NaN")

In [ ]:
# Visualisation des méthodes d'interpolation
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for ax, method in zip(axes.flat, ['linear', 'nearest', 'cubic', 'zero']):
    result = converter.interpolate_to_higher_frequency(series_q, 'MS', method=method)

    ax.plot(result.index, result.values, '-', color=COLORS['interpolated'],
            alpha=0.7, label=f'Mensuel ({method})')
    ax.plot(series_q.index, series_q.values, 'o', color=COLORS['original'],
            markersize=8, label='Trimestriel (original)')

    ax.set_title(f"method='{method}'")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('Interpolation trimestrielle → mensuelle : comparaison des méthodes', fontsize=12)
plt.tight_layout()
plt.show()

### 5.2 Impact du paramètre `limit`

`limit` contrôle le nombre maximum de NaN consécutifs remplis par l'interpolation :
- `limit='default'` → utilise le facteur de conversion (3 pour Q→M)
- `limit=None` → pas de limite, remplit tous les NaN
- `limit=1` → ne remplit qu'un seul NaN consécutif

In [ ]:
print_section("Impact de limit sur l'interpolation Q → M")

for limit_val in ['default', None, 1, 2]:
    result = converter.interpolate_to_higher_frequency(
        series_q, 'MS', method='linear', limit=limit_val
    )
    nan_count = result.isna().sum()
    print(f"  limit={str(limit_val):>9s} : {len(result)} points | {nan_count} NaN")
    # Affichage d'un extrait
    print(f"    Extrait : {result.iloc[:6].values.round(2).tolist()}")

### 5.3 Impact du paramètre `limit_direction`

`limit_direction` contrôle la direction de propagation de l'interpolation :
- `'forward'` → propage vers l'avant (défaut pour position `'S'`)
- `'backward'` → propage vers l'arrière (défaut pour position `'E'`)
- `'both'` → propage dans les deux directions

In [ ]:
print_section("Impact de limit_direction (Q → M)")

for direction in ['forward', 'backward', 'both']:
    result = converter.interpolate_to_higher_frequency(
        series_q, 'MS', method='linear',
        limit='default', limit_direction=direction
    )
    nan_count = result.isna().sum()
    print(f"  limit_direction='{direction:>8s}' : {nan_count} NaN")
    print(f"    Début : {result.iloc[:4].values.round(2).tolist()}")
    print(f"    Fin   : {result.iloc[-4:].values.round(2).tolist()}")

### 5.4 Impact du paramètre `limit_area`

`limit_area` restreint les zones où l'interpolation s'applique :
- `'inside'` → uniquement entre valeurs connues (pas d'extrapolation)
- `'outside'` → uniquement aux extrémités
- `None` → partout

In [ ]:
# Série avec NaN au milieu pour illustrer inside/outside
dates_q_holes = pd.date_range('2024-01-01', periods=8, freq='QS')
values_holes = [100, np.nan, 115, 130, 125, np.nan, 135, 150]
series_holes = pd.Series(values_holes, index=dates_q_holes, name='pib')

print("Série trimestrielle avec NaN internes :")
print(series_holes)

print_section("Impact de limit_area (Q → M)")

for area in [None, 'inside', 'outside']:
    result = converter.interpolate_to_higher_frequency(
        series_holes, 'MS', method='linear', limit=None, limit_area=area
    )
    nan_count = result.isna().sum()
    print(f"  limit_area={str(area):>9s} : {nan_count} NaN / {len(result)} points")

In [ ]:
# Visualisation comparative des limit_area
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, area in zip(axes, [None, 'inside', 'outside']):
    result = converter.interpolate_to_higher_frequency(
        series_holes, 'MS', method='linear', limit=None, limit_area=area
    )
    mask_valid = result.notna()
    mask_nan = result.isna()

    ax.plot(result.index[mask_valid], result.values[mask_valid], '-',
            color=COLORS['interpolated'], alpha=0.7, label='Interpolé')
    if mask_nan.any():
        ax.plot(result.index[mask_nan], [0] * mask_nan.sum(), 'x',
                color=COLORS['nan'], markersize=6, label='NaN restants')
    ax.plot(series_holes.dropna().index, series_holes.dropna().values, 'o',
            color=COLORS['original'], markersize=8, label='Original')

    label = str(area) if area else 'None'
    ax.set_title(f"limit_area='{label}'")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

fig.suptitle("Impact de limit_area sur l'interpolation", fontsize=12)
plt.tight_layout()
plt.show()

### 5.5 Impact de la position cible sur la direction par défaut

La position est portée par la fréquence cible (`MS`/`ME`). Quand `limit_direction` n'est pas spécifié :
- position start (`MS`) → `limit_direction='forward'` (propagation vers l'avant)
- position end (`ME`) → `limit_direction='backward'` (propagation vers l'arrière)


In [ ]:
print_section("Impact de target_position sur la direction implicite")

# Série trimestrielle QS
series_qs = pd.Series(values_q, index=pd.date_range('2024-01-01', periods=8, freq='QS'), name='pib')

for target_pos in ['S', 'E']:
    target_freq = 'MS' if target_pos == 'S' else 'ME'
    result = converter.interpolate_to_higher_frequency(
        series_qs, target_freq, method='linear',
        limit='default'
    )
    nan_count = result.isna().sum()
    print(f"  position '{target_pos}' (via target_freq='{target_freq}') : {nan_count} NaN")
    # Premier trimestre détaillé
    first_6 = result.iloc[:6]
    print(f"    Premier trimestre : {[f'{d.strftime("%Y-%m-%d")}={v:.1f}' if not np.isnan(v) else f'{d.strftime("%Y-%m-%d")}=NaN' for d, v in first_6.items()]}")

### 5.6 NaN en début de période et délais de publication en fin de période

Scénario réaliste :
- Les **2 premiers trimestres** ont des NaN (données non disponibles)
- Les **2 derniers trimestres** ont des NaN (délais de publication)

In [ ]:
# Série trimestrielle avec NaN en début et fin
dates_q_pub = pd.date_range('2023-01-01', periods=10, freq='QS')
values_pub = [np.nan, np.nan, 105, 120, 115, 130, 125, 140, np.nan, np.nan]
series_pub = pd.Series(values_pub, index=dates_q_pub, name='pib_delayed')

print("Série trimestrielle avec NaN (délais de publication) :")
print(series_pub)

print_section("Interpolation avec NaN : impact de limit_area")

results_pub = {}
for area in [None, 'inside', 'outside']:
    result = converter.interpolate_to_higher_frequency(
        series_pub, 'MS', method='linear', limit=None, limit_area=area
    )
    results_pub[str(area)] = result
    nan_count = result.isna().sum()
    label = str(area) if area else 'None'
    print(f"  limit_area='{label}' : {nan_count} NaN / {len(result)} points")

In [ ]:
# Visualisation comparative
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

fill_colors = {'None': COLORS['interpolated'], 'inside': COLORS['ffill'], 'outside': COLORS['bfill']}

for ax, (label, result) in zip(axes, results_pub.items()):
    color = fill_colors[label]
    mask_valid = result.notna()
    mask_nan = result.isna()

    ax.plot(result.index[mask_valid], result.values[mask_valid], '-',
            color=color, alpha=0.7, label='Interpolé')
    if mask_nan.any():
        ax.scatter(result.index[mask_nan], [result.min() - 5] * mask_nan.sum(),
                   marker='v', color=COLORS['nan'], s=20, label='NaN', alpha=0.5)

    ax.plot(series_pub.dropna().index, series_pub.dropna().values, 'o',
            color=COLORS['original'], markersize=8, zorder=5, label='Original')

    # Zones NaN en fond
    ax.axvspan(dates_q_pub[0], dates_q_pub[1], alpha=0.08, color='red', label='Zone NaN début')
    ax.axvspan(dates_q_pub[-2], dates_q_pub[-1] + pd.DateOffset(months=3), alpha=0.08, color='orange', label='Zone NaN fin')

    ax.set_title(f"limit_area='{label}'")
    ax.legend(fontsize=7, loc='upper left')
    ax.grid(True, alpha=0.3)

fig.suptitle("Interpolation Q→M avec NaN en début et délais de publication en fin", fontsize=12)
plt.tight_layout()
plt.show()

### 5.7 Tableau récapitulatif des NaN par zone

| `limit_area` | NaN en début (pas de données) | Zone centrale (données disponibles) | NaN en fin (délai de publication) |
|---|---|---|---|
| `None` | Interpolés si possible | Tous remplis | Interpolés si possible |
| `'inside'` | Restent NaN | Tous remplis | Restent NaN |
| `'outside'` | Remplis (extrapolation) | Restent NaN entre les points | Remplis (extrapolation) |

**Recommandation** : pour les données économiques avec délais de publication, `limit_area='inside'` évite l'extrapolation hasardeuse aux extrémités.

### 5.8 Données de panel

In [ ]:
# Panel trimestriel avec profils de NaN asymétriques
dates_q_panel = pd.date_range('2023-01-01', periods=8, freq='QS')

# Entité A : NaN en fin (délai de publication)
vals_pa = [100, 110, 105, 120, 115, 130, np.nan, np.nan]
df_qa = pd.DataFrame({'pib': vals_pa}, index=dates_q_panel)
df_qa['entity'] = 'entité_A'

# Entité B : NaN en début (série courte historiquement)
vals_pb = [np.nan, np.nan, 200, 210, 205, 220, 215, 230]
df_qb = pd.DataFrame({'pib': vals_pb}, index=dates_q_panel)
df_qb['entity'] = 'entité_B'

# Assemblage du panel
df_panel_interp = pd.concat([df_qa, df_qb]).set_index('entity', append=True).swaplevel()
df_panel_interp.index.names = ['entity', 'date']

print("Panel trimestriel (NaN asymétriques) :")
print(df_panel_interp)

# Interpolation par entité
print_section("Interpolation panel Q → M avec limit_area='inside'")

for entity in ['entité_A', 'entité_B']:
    sub = df_panel_interp.xs(entity, level='entity')
    result = converter.interpolate_to_higher_frequency(
        sub, 'MS', method='linear', limit=None, limit_area='inside'
    )
    nan_count = result['pib'].isna().sum()
    print(f"\n  {entity} ({nan_count} NaN restants) :")
    print(result.to_string(float_format='{:.2f}'.format))

In [ ]:
# Visualisation panel
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, entity in zip(axes, ['entité_A', 'entité_B']):
    sub_orig = df_panel_interp.xs(entity, level='entity')

    for area, color, ls in [('inside', COLORS['ffill'], '-'), (None, COLORS['interpolated'], '--')]:
        result = converter.interpolate_to_higher_frequency(
            sub_orig, 'MS', method='linear', limit=None, limit_area=area
        )
        mask = result['pib'].notna()
        label_area = str(area) if area else 'None'
        ax.plot(result.index[mask], result['pib'].values[mask], ls,
                color=color, alpha=0.7, label=f"limit_area='{label_area}'")

    ax.plot(sub_orig.dropna().index, sub_orig.dropna()['pib'].values, 'o',
            color=COLORS['original'], markersize=8, label='Original', zorder=5)

    ax.set_title(entity)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle("Interpolation panel Q→M : comparaison limit_area par entité", fontsize=12)
plt.tight_layout()
plt.show()

---
## 6. Alignement multi-sources avec `convert_frequency`

Pour aligner plusieurs jeux de données de fréquences différentes sur une fréquence commune,
il suffit d'appliquer `convert_frequency` à chacun d'eux avec la même fréquence cible.
(L'ancienne méthode `align_frequencies` a été supprimée : pour la construction de jeux de
données homogènes destinés au `HighFrequencyImputer`, voir le `FrequencyAligner`.)


### 6.1 Séries temporelles

In [ ]:
# Série journalière
dates_daily = pd.date_range('2024-01-01', periods=180, freq='D')
series_daily = pd.Series(
    np.sin(np.linspace(0, 4 * np.pi, 180)) * 20 + 100,
    index=dates_daily, name='temperature'
)

# Série mensuelle
dates_monthly = pd.date_range('2024-01-01', periods=6, freq='MS')
series_monthly = pd.Series([50, 55, 60, 58, 62, 65], index=dates_monthly, name='indice_confiance')

# Série trimestrielle
dates_quarterly = pd.date_range('2024-01-01', periods=2, freq='QS')
series_quarterly = pd.Series([1.2, 1.5], index=dates_quarterly, name='pib_growth')

print("Avant alignement :")
print(f"  Journalier   : {len(series_daily)} points | {series_daily.index[0].date()} → {series_daily.index[-1].date()}")
print(f"  Mensuel      : {len(series_monthly)} points | {series_monthly.index[0].date()} → {series_monthly.index[-1].date()}")
print(f"  Trimestriel  : {len(series_quarterly)} points | {series_quarterly.index[0].date()} → {series_quarterly.index[-1].date()}")

# Alignement vers mensuel : conversion de chaque série vers la fréquence commune.
# La méthode dépend de la direction : agrégation ('mean') en downsampling,
# interpolation ('linear') en upsampling
aligned = [
    converter.convert_frequency(series_daily, 'monthly', method='mean'),
    converter.convert_frequency(series_monthly, 'monthly', method='mean'),
    converter.convert_frequency(series_quarterly, 'monthly', method='linear'),
]

print("\nAprès alignement (target_freq='monthly') :")
for i, (name, s) in enumerate(zip(['temperature', 'indice_confiance', 'pib_growth'], aligned)):
    print(f"  {name:>20s} : {len(s)} points")
    print(f"    {s.to_string(float_format='{:.2f}'.format)}")
    print()


In [ ]:
# Visualisation avant / après alignement
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Avant
ax = axes[0]
ax.plot(series_daily.index, series_daily.values, '-', color=COLORS['original'],
        alpha=0.5, label='Journalier')
ax.plot(series_monthly.index, series_monthly.values, 's-', color=COLORS['aggregated'],
        markersize=8, label='Mensuel')
ax2 = ax.twinx()
ax2.plot(series_quarterly.index, series_quarterly.values, 'D-', color=COLORS['interpolated'],
         markersize=10, label='Trimestriel')
ax.set_title('Avant alignement')
ax.legend(loc='upper left', fontsize=8)
ax2.legend(loc='upper right', fontsize=8)
ax.grid(True, alpha=0.3)

# Après
ax = axes[1]
for s, name, color, marker in zip(aligned,
    ['temperature (D→M)', 'indice_confiance (M)', 'pib_growth (Q→M)'],
    [COLORS['original'], COLORS['aggregated'], COLORS['interpolated']],
    ['o', 's', 'D']):
    ax.plot(s.index, s.values, f'{marker}-', color=color, markersize=6,
            alpha=0.7, label=name)

ax.set_title('Après alignement (mensuel)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

fig.suptitle("Alignement multi-fréquences par conversions successives", fontsize=12)
plt.tight_layout()
plt.show()

### 6.2 Alignement automatique (choix de la fréquence commune)

Sans fréquence cible imposée, on choisit la fréquence **la moins granulaire** parmi les
fréquences détectées (via `detect_frequency` et `get_frequency_order`) : c'est la seule vers
laquelle toutes les séries peuvent être agrégées sans invention de données.


In [ ]:
# Détection des fréquences et choix de la moins granulaire comme cible commune
from tsforecast.frequency.detector import detect_frequency
from tsforecast.utils.frequency.utils import get_frequency_order

series_list = [series_daily, series_monthly, series_quarterly]
detected = [detect_frequency(s, return_format='base') for s in series_list]
target_auto = max(detected, key=get_frequency_order)

print(f"Fréquences détectées : {detected} → cible commune : {target_auto}")

# Conversion de chaque série vers la fréquence commune
aligned_auto = [
    converter.convert_frequency(s, target_auto, method='mean')
    for s in series_list
]

print("\nAlignement automatique :")
for s in aligned_auto:
    freq = s.index.inferred_freq if len(s) >= 3 else 'n/a (moins de 3 points)'
    print(f"  Fréquence résultante : {freq} | {len(s)} points")


### 6.3 Données de panel

In [ ]:
# Deux panels de fréquences différentes
# Panel mensuel
dates_pm3 = pd.date_range('2024-01-01', periods=6, freq='MS')
frames_m = []
for entity in ['FR', 'DE']:
    df_e = pd.DataFrame({'exports': rng.normal(100, 5, 6)}, index=dates_pm3)
    df_e['pays'] = entity
    frames_m.append(df_e)
panel_monthly = pd.concat(frames_m).set_index('pays', append=True).swaplevel()
panel_monthly.index.names = ['pays', 'date']

# Panel trimestriel
dates_pq = pd.date_range('2024-01-01', periods=2, freq='QS')
frames_q = []
for entity in ['FR', 'DE']:
    df_e = pd.DataFrame({'pib': rng.normal(500, 20, 2)}, index=dates_pq)
    df_e['pays'] = entity
    frames_q.append(df_e)
panel_quarterly = pd.concat(frames_q).set_index('pays', append=True).swaplevel()
panel_quarterly.index.names = ['pays', 'date']

print("Panel mensuel :")
print(panel_monthly)
print("\nPanel trimestriel :")
print(panel_quarterly)

# Alignement vers trimestriel : conversion de chaque panel vers la fréquence commune
aligned_panels = [
    converter.convert_frequency(p, 'quarterly', method='sum')
    for p in (panel_monthly, panel_quarterly)
]

print("\nPanels alignés (trimestriel, sum) :")
for i, p in enumerate(aligned_panels):
    print(f"\n  Dataset {i+1} :")
    print(p)

---
## 7. Synthèse

### Choix de la méthode

| Objectif | Méthode recommandée |
|---|---|
| Conversion simple (API unifiée) | `convert()` |
| Conversion avec contrôle fin | `convert_frequency()` |
| Downsampling avec gestion des périodes incomplètes | `aggregate_to_lower_frequency(full_periods_only=True)` |
| Upsampling avec contrôle de l'extrapolation | `interpolate_to_higher_frequency(limit_area='inside')` |
| Alignement multi-sources | `convert_frequency()` sur chaque dataset |
| Datasets pour le `HighFrequencyImputer` (index préservé/densifié) | `FrequencyAligner` |
| Estimation du ratio fréquentiel | `get_conversion_factor()` |

### Paramètres clés

| Paramètre | Contexte | Effet |
|---|---|---|
| `full_periods_only` | Agrégation | Masque les périodes incomplètes (NaN) |
| `limit` | Interpolation | Contrôle le nombre de NaN consécutifs remplis |
| `limit_direction` | Interpolation | Direction de propagation (`forward`/`backward`/`both`) |
| `limit_area` | Interpolation | Zone d'application (`inside`/`outside`/`None`) |
| `target_position` | Les deux | Position de l'index cible (`S`/`E`) |
| `alignment_method` | `convert_frequency` (DataFrame) | Alignement lors de fréquences mixtes |